# Module 2 · Inside a convolutional network
Run every cell from top to bottom (**Run → Run All Cells**). Nothing to code: this notebook shows, with a real network,
what the Module 2 class explains. The first run downloads ResNet-18 (45 MB) and YOLO26n (5 MB) once.

Change `PHOTO` to one of **your** photos and run it again: that is how you learn what the layers see.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import numpy as np, cv2, torch, torchvision, matplotlib.pyplot as plt
from PIL import Image
from torchvision.models import ResNet18_Weights
W = ResNet18_Weights.IMAGENET1K_V1
net = torchvision.models.resnet18(weights=W).eval()          # a CNN trained on 1.2 million ImageNet photos
prep, CATS = W.transforms(), W.meta["categories"]
PHOTO = "data/m2_samples/pothole.png"                      # change me
img = np.array(Image.open(PHOTO).convert("RGB"))
x = prep(Image.fromarray(img)).unsqueeze(0)
plt.imshow(img); plt.axis("off"); plt.title(PHOTO); plt.show()

## 1. The filters were learned, not written
Module 1: we wrote 3×3 kernels by hand. Here are the 64 filters (7×7, colour) of the **first layer**, learned from data.

In [ ]:
w = net.conv1.weight.detach().numpy()
fig, axes = plt.subplots(8, 8, figsize=(7, 7))
for i, ax in enumerate(axes.flat):
    f = w[i].transpose(1, 2, 0); ax.imshow((f - f.min()) / (f.max() - f.min())); ax.axis("off")
plt.suptitle("64 learned filters of the first layer"); plt.show()

## 2. The path of one image through the network
Each stage: convolution → ReLU → (pooling). The maps get **smaller** and **more numerous**.

In [ ]:
acts = {}
with torch.no_grad():
    y = net.relu(net.bn1(net.conv1(x))); acts["first layer"] = y
    y = net.maxpool(y); acts["after pooling"] = y
    for n in ("layer1", "layer2", "layer3", "layer4"):
        y = getattr(net, n)(y); acts[n] = y
    logits = net.fc(net.avgpool(y).flatten(1))
print(f"{'input':14s} {tuple(x.shape[1:])}")
for k, a in acts.items():
    print(f"{k:14s} {tuple(a.shape[1:])}   (channels, height, width)")
print(f"{'classifier':14s} {tuple(logits.shape[1:])}   one score per ImageNet class")

In [ ]:
for k, a in acts.items():
    t = a[0].numpy(); idx = np.argsort(-t.reshape(t.shape[0], -1).mean(1))[:8]
    fig, axes = plt.subplots(1, 8, figsize=(16, 2.2))
    for ax, i in zip(axes, idx): ax.imshow(t[i], cmap="magma"); ax.axis("off")
    fig.suptitle(f"{k}: 8 of {t.shape[0]} maps, each {t.shape[1]}×{t.shape[2]}"); plt.show()

**Look:** the first maps still look like the photo (edges, texture). The deep ones are tiny blobs: they no longer say
*where an edge is*, they say *whether a part of an object is there*. That is what “early layers learn simple patterns,
deep layers learn parts and objects” means.

## 3. ReLU and pooling, with numbers

In [ ]:
a = np.array([[-3, 2], [5, -1]]); print("ReLU keeps positives, negatives become 0:\n", a, "->\n", np.maximum(a, 0))
g = np.array([[1, 3, 2, 0], [4, 6, 1, 1], [0, 2, 9, 5], [1, 1, 3, 7]])
print("\nMax pooling 2×2 keeps the largest value of each block:\n", g, "->\n", g.reshape(2, 2, 2, 2).max(axis=(1, 3)))

## 4. Classification, localisation, detection
Same photo, three different questions.

In [ ]:
STREET = "data/m2_samples/street.jpg"                    # change me
s = np.array(Image.open(STREET).convert("RGB"))
with torch.no_grad():
    p = torch.softmax(net(prep(Image.fromarray(s)).unsqueeze(0)), 1)[0]
top = torch.topk(p, 5)
print("CLASSIFICATION (ResNet-18): one label for the whole image")
for v, i in zip(top.values, top.indices): print(f"  {CATS[i]:20s} {v:.2f}")

In [ ]:
from ultralytics import YOLO
yolo = YOLO("yolo26n.pt")                                 # a detector trained on COCO (80 classes)
r = yolo.predict(s[:, :, ::-1].copy(), conf=0.25, imgsz=960, verbose=False)[0]
names = [r.names[int(c)] for c in r.boxes.cls]
print("DETECTION (YOLO26n): every object, each with a class, a box and a confidence")
for n, c in zip(names, r.boxes.conf): print(f"  {n:12s} {c:.2f}")
plt.figure(figsize=(12, 7)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off"); plt.show()

**Your turn (activity, Part 1):** on paper, draw the path of one image through a small CNN:
input → conv + ReLU → pool → conv + ReLU → pool → classifier, writing the size at every step.
Then name, for your field, one task a CNN already does well and one it should not be trusted with.